RECOPILACION DE DATOS

In [1]:
# Ejecuta esta celda primero en un kernel nuevo para instalar dependencias.
%pip -q install amplpy pandas numpy openpyxl scipy matplotlib seaborn yfinance ipykernel

from pathlib import Path

import yfinance as yf
import pandas as pd

def _resolver_data_dir():
    notebook_file = globals().get("__vsc_ipynb_file__")
    if notebook_file:
        return Path(notebook_file).expanduser().resolve().parent
    return Path.cwd().resolve()

# En VS Code/Jupyter local se guarda y lee desde la carpeta del notebook.
# Si quieres usar otra carpeta, cambia DATA_DIR_MANUAL por una ruta como Path(r"C:\Users\tu_usuario\datos").
DATA_DIR_MANUAL = None
DATA_DIR = Path(DATA_DIR_MANUAL).expanduser().resolve() if DATA_DIR_MANUAL else _resolver_data_dir()
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Carpeta de trabajo local: {DATA_DIR}")

# 1. Definir la lista de acciones
tickers = [
    "COPEC.SN", "SQM-B.SN", "ENELAM.SN", "AGUAS-A.SN",
    "CHILE.SN", "CENCOSUD.SN", "SMU.SN", "PARAUCO.SN",
    "MALLPLAZA.SN", "CMPC.SN"
]

# 2. Fechas
fecha_inicio = "2019-12-01"
fecha_fin = "2026-01-01"

# 3. Descargar datos semanales
print("Descargando datos históricos semanales de Yahoo Finance...")
# auto_adjust=False asegura que descargue la columna 'Adj Close' por separado
datos = yf.download(tickers, start=fecha_inicio, end=fecha_fin, interval="1wk", auto_adjust=False)

# 4. Extraer precios de cierre ajustado y calcular retornos
# o simplemente filtramos por el nivel de precios si es un MultiIndex.
if 'Adj Close' in datos.columns.levels[0]:
    precios = datos['Adj Close']
else:
    # Fallback para estructuras donde el precio ajustado se descarga como Close con auto_adjust
    precios = datos['Close']

retornos_semanales = precios.pct_change().dropna()

# Transponer la matriz para que las acciones queden en las filas y las fechas en las columnas
retornos_semanales = retornos_semanales.T

# --- MOSTRAR RESULTADOS ---
print("\n--- Retornos Semanales (Todos los meses) ---")
print(retornos_semanales.to_string())

# --- GUARDAR LOS RESULTADOS ---
nombre_archivo_excel = DATA_DIR / "Retornos_Semanales_Acciones_Chilenas.xlsx"
nombre_archivo_csv = DATA_DIR / "Retornos_Semanales_Acciones_Chilenas.csv"

retornos_semanales.to_csv(nombre_archivo_csv)
print(f"\n¡Listo! Guardado en: {nombre_archivo_csv}")

try:
    retornos_semanales.to_excel(nombre_archivo_excel)
    print(f"¡Guardado en Excel: {nombre_archivo_excel}!")
except Exception:
    print("Nota: No se pudo guardar en Excel (posiblemente falta openpyxl).")


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Carpeta de trabajo local: C:\Users\brand\Downloads
Descargando datos históricos semanales de Yahoo Finance...


[*********************100%***********************]  10 of 10 completed



--- Retornos Semanales (Todos los meses) ---
Date          2019-12-09  2019-12-16  2019-12-23  2019-12-30  2020-01-06  2020-01-13  2020-01-20  2020-01-27  2020-02-03  2020-02-10  2020-02-17  2020-02-24  2020-03-02  2020-03-09  2020-03-16  2020-03-23  2020-03-30  2020-04-06  2020-04-13  2020-04-20  2020-04-27  2020-05-04  2020-05-11  2020-05-18  2020-05-25  2020-06-01  2020-06-08  2020-06-15  2020-06-22  2020-06-29  2020-07-06  2020-07-13  2020-07-20  2020-07-27  2020-08-03  2020-08-10  2020-08-17  2020-08-24  2020-08-31  2020-09-07  2020-09-14  2020-09-21  2020-09-28  2020-10-05  2020-10-12  2020-10-19  2020-10-26  2020-11-02  2020-11-09  2020-11-16  2020-11-23  2020-11-30  2020-12-07  2020-12-14  2020-12-21  2020-12-28  2021-01-04  2021-01-11  2021-01-18  2021-01-25  2021-02-01  2021-02-08  2021-02-15  2021-02-22  2021-03-01  2021-03-08  2021-03-15  2021-03-22  2021-03-29  2021-04-05  2021-04-12  2021-04-19  2021-04-26  2021-05-03  2021-05-10  2021-05-17  2021-05-24  2021-05-31  2021

In [2]:
# --- Verificacion de espaciado semanal consistente (evidencia para la defensa) ---
fechas = precios.index.to_series().sort_index()
gaps_dias = fechas.diff().dt.days.dropna()

print("Distribucion de gaps entre observaciones consecutivas (en dias):")
print(gaps_dias.value_counts().sort_index())

gaps_anomalos = gaps_dias[gaps_dias != 7]
if len(gaps_anomalos) > 0:
    print(f"\nATENCION: {len(gaps_anomalos)} gaps distintos de 7 dias detectados:")
    for fecha, gap in gaps_anomalos.items():
        print(f"  {fecha.date()}: gap de {gap} dias respecto a la observacion anterior")
else:
    print("\nOK: todas las observaciones estan separadas exactamente por 7 dias.")

print(f"\nTotal observaciones: {len(fechas)}")
print(f"Rango: {fechas.min().date()} a {fechas.max().date()}")

Distribucion de gaps entre observaciones consecutivas (en dias):
Date
7.0    317
Name: count, dtype: int64

OK: todas las observaciones estan separadas exactamente por 7 dias.

Total observaciones: 318
Rango: 2019-12-02 a 2025-12-29
